In [ ]:
import os, glob, cv2, numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import torch.nn.functional as F

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

In [3]:
class SatelliteSegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None, max_samples=None):
        self.images_paths = sorted(glob.glob(os.path.join(images_dir, "*.png")))
        self.masks_paths = sorted(glob.glob(os.path.join(masks_dir, "*.png")))
        if max_samples is not None:
            self.images_paths = self.images_paths[:max_samples]
            self.masks_paths = self.masks_paths[:max_samples]
        self.transform = transform

    def __len__(self):
        return len(self.images_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.images_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'].unsqueeze(0)
        else:
            image = torch.tensor(image).permute(2,0,1).float()/255.0
            mask = torch.tensor(mask).unsqueeze(0).float()
        return image, mask

In [ ]:
train_transform = A.Compose([
    A.Resize(256,256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5)),
    ToTensorV2()
])

In [5]:
train_transform_enh = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.RandomGamma(p=0.5),
    A.Resize(256, 256),
    ToTensorV2()
])

val_transform_enh = A.Compose([
    A.Resize(256, 256),
    ToTensorV2()
])

In [6]:
MAX_IMG_COUNT = 5

In [ ]:
train_dataset = SatelliteSegmentationDataset(
    images_dir="data/training/images",
    masks_dir="data/training/groundtruth",
    transform=train_transform,
    # max_samples= MAX_IMG_COUNT
)
gen_dataset = SatelliteSegmentationDataset(
    images_dir="data/training/images_generated",
    masks_dir="data/training/groundtruth_generated",
    transform=train_transform,
    # max_samples= MAX_IMG_COUNT
)
full_train_dataset = torch.utils.data.ConcatDataset([train_dataset, gen_dataset])
train_loader = DataLoader(full_train_dataset, batch_size=8, shuffle=True, num_workers=0)

In [ ]:
train_dataset_enh_orig = SatelliteSegmentationDataset(
    images_dir="data/training/images",
    masks_dir="data/training/groundtruth",
    transform=train_transform_enh,
    # max_samples=MAX_IMG_COUNT
)

train_dataset_enh_gen = SatelliteSegmentationDataset(
    images_dir="data/training/images_generated",
    masks_dir="data/training/groundtruth_generated",
    transform=train_transform_enh,
    # max_samples=MAX_IMG_COUNT
)

full_train_dataset_enh = torch.utils.data.ConcatDataset([train_dataset_enh_orig, train_dataset_enh_gen])
train_loader_enh = DataLoader(full_train_dataset_enh, batch_size=8, shuffle=True, num_workers=0)

In [9]:
def dice_coeff(pred, target, smooth=1e-5):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    return (2.*intersection + smooth)/(pred.sum()+target.sum()+smooth)

def iou_score(pred, target, smooth=1e-5):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum()+target.sum()-intersection
    return (intersection + smooth)/(union + smooth)

def dice_bce_loss(preds, targets):
    bce = torch.nn.functional.binary_cross_entropy_with_logits(preds, targets)
    preds_sigmoid = torch.sigmoid(preds)
    intersection = (preds_sigmoid * targets).sum()
    dice = (2. * intersection + 1e-7) / (preds_sigmoid.sum() + targets.sum() + 1e-7)
    return bce + (1 - dice)

def precision_score(pred, target, smooth=1e-5):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    return (tp + smooth) / (tp + fp + smooth)

def recall_score(pred, target, smooth=1e-5):
    pred = torch.sigmoid(pred)
    pred = (pred > 0.5).float()
    tp = (pred * target).sum()
    fn = ((1 - pred) * target).sum()
    return (tp + smooth) / (tp + fn + smooth)

In [10]:
def train_epoch(loader, model, criterion, optimizer, device):
    model.train()
    running_loss = 0
    for images, masks in tqdm(loader):
        images = images.to(device).float()
        masks = masks.to(device).float()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()*images.size(0)
    return running_loss/len(loader.dataset)

def eval_epoch(loader, model, device):
    model.eval()
    dices, ious, precisions, recalls = [], [], [], []
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device).float()
            masks = masks.to(device).float()
            outputs = model(images)
            dices.append(dice_coeff(outputs, masks).item())
            ious.append(iou_score(outputs, masks).item())
            precisions.append(precision_score(outputs, masks).item())
            recalls.append(recall_score(outputs, masks).item())
    return np.mean(dices), np.mean(ious), np.mean(precisions), np.mean(recalls)

In [11]:
results = {}

In [ ]:
cnn_baseline = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)
cnn_criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
cnn_optimizer = torch.optim.Adam(cnn_baseline.parameters(), lr=3e-3)

print("=== Training CNN Baseline ===")
for epoch in range(10):
    loss = train_epoch(train_loader, cnn_baseline, cnn_criterion, cnn_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, cnn_baseline, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

results["CNN Baseline"] = (dice, iou, precision, recall)

In [ ]:
cnn_enhanced = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
).to(device)

cnn_enh_optimizer = torch.optim.AdamW(cnn_enhanced.parameters(), lr=1e-4)

cnn_enh_criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)

print("=== Training CNN Enhanced ===")
for epoch in range(10):
    loss = train_epoch(train_loader, cnn_enhanced, cnn_enh_criterion, cnn_enh_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, cnn_enhanced, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

results["CNN Enhanced"] = (dice, iou, precision, recall)

In [ ]:
vit_baseline = smp.Unet(
    encoder_name="mit_b0",
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1
).to(device)
vit_optimizer = torch.optim.Adam(vit_baseline.parameters(), lr=3e-3)

print("=== Training ViT Baseline ===")
for epoch in range(10):
    loss = train_epoch(train_loader, vit_baseline, dice_bce_loss, vit_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, vit_baseline, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
results["ViT Baseline"] = (dice, iou, precision, recall)

In [ ]:
vit_enhanced = smp.Unet(
    encoder_name="mit_b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_channels=[256, 128, 64, 32, 16],
    decoder_attention_type='scse'
).to(device)

vit_enh_optimizer = torch.optim.Adam(vit_enhanced.parameters(), lr=1e-3)

print("=== Training ViT Enhanced ===")
for epoch in range(10):
    loss = train_epoch(train_loader_enh, vit_enhanced, dice_bce_loss, vit_enh_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, vit_enhanced, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
results["ViT Enhanced"] = (dice, iou, precision, recall)

In [16]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()

        self.input_norm = nn.BatchNorm2d(n_channels)

        self.dconv_down1 = DoubleConv(n_channels, 64)
        self.dconv_down2 = DoubleConv(64, 128)
        self.dconv_down3 = DoubleConv(128, 256)
        self.dconv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.dconv_up3 = DoubleConv(256+512, 256)
        self.dconv_up2 = DoubleConv(128+256, 128)
        self.dconv_up1 = DoubleConv(128+64, 64)

        self.conv_last = nn.Conv2d(64, n_classes, 1)

    def forward(self, x):
        x = self.input_norm(x)

        conv1 = self.dconv_down1(x)
        conv2 = self.dconv_down2(self.maxpool(conv1))
        conv3 = self.dconv_down3(self.maxpool(conv2))
        conv4 = self.dconv_down4(self.maxpool(conv3))

        x = self.upsample(conv4)
        x = torch.cat([x, conv3], dim=1)
        x = self.dconv_up3(x)

        x = self.upsample(x)
        x = torch.cat([x, conv2], dim=1)
        x = self.dconv_up2(x)

        x = self.upsample(x)
        x = torch.cat([x, conv1], dim=1)
        x = self.dconv_up1(x)

        x = self.conv_last(x)
        return x

In [ ]:
own_cnn = UNet().to(device)
own_cnn_optimizer = torch.optim.Adam(own_cnn.parameters(), lr=3e-3)
own_cnn_criterion = nn.BCEWithLogitsLoss()

print("=== Training Own CNN ===")
for epoch in range(10):
    loss = train_epoch(train_loader, own_cnn, own_cnn_criterion, own_cnn_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, own_cnn, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
results["Own CNN"] = (dice, iou, precision, recall)

In [ ]:
own_cnn_enh = UNet().to(device)
own_cnn_enh_optimizer = torch.optim.Adam(own_cnn_enh.parameters(), lr=1e-3)

print("=== Training Own CNN Enhanced ===")
for epoch in range(10):
    loss = train_epoch(train_loader, own_cnn_enh, own_cnn_criterion, own_cnn_enh_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, own_cnn_enh, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
results["Own CNN Enhanced"] = (dice, iou, precision, recall)

In [19]:

class SimpleViTSegmenter(nn.Module):
    def __init__(self, img_size=256, patch_size=8, in_channels=3, embed_dim=256, num_classes=1, nhead=8, num_layers=4):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.num_classes = num_classes

        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm_patch = nn.LayerNorm(embed_dim)

        self.max_patches_h = img_size // patch_size
        self.max_patches_w = img_size // patch_size
        self.max_patches = self.max_patches_h * self.max_patches_w
        self.pos_embed = nn.Parameter(torch.randn(1, self.max_patches, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim*4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm_transformer = nn.LayerNorm(embed_dim)

        self.decode_conv = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, x):
        B, C, H, W = x.shape

        x = self.patch_embed(x)
        H_p, W_p = x.shape[2], x.shape[3]
        num_patches = H_p * W_p

        x = x.flatten(2).transpose(1, 2)
        x = self.norm_patch(x)

        pos_embed_resized = self.pos_embed[:, :num_patches, :].repeat(B, 1, 1)
        x = x + pos_embed_resized

        x = self.transformer(x)
        x = self.norm_transformer(x)

        x = x.transpose(1, 2).view(B, self.embed_dim, H_p, W_p)

        x = F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False)

        x = self.decode_conv(x)
        x = torch.sigmoid(x)

        return x


In [ ]:
own_vit = SimpleViTSegmenter().to(device)
own_vit_optimizer = torch.optim.Adam(own_vit.parameters(), lr=1e-3)
own_vit_criterion = nn.BCEWithLogitsLoss()

print("=== Training Own ViT ===")
for epoch in range(10):
    loss = train_epoch(train_loader, own_vit, own_vit_criterion, own_vit_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, own_vit, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
results["Own ViT"] = (dice, iou, precision, recall)

In [ ]:
own_vit_enh = SimpleViTSegmenter(patch_size=8, embed_dim=256).to(device)
own_vit_enh_optimizer = torch.optim.Adam(own_vit_enh.parameters(), lr=3e-3)

print("=== Training Own ViT Enhanced ===")
for epoch in range(10):
    loss = train_epoch(train_loader_enh, own_vit_enh, dice_bce_loss, own_vit_enh_optimizer, device)
    dice, iou, precision, recall = eval_epoch(train_loader, own_vit_enh, device)
    print(f"Dice: {dice:.4f} | IoU: {iou:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")
results["Own ViT Enhanced"] = (dice, iou, precision, recall)

In [ ]:
print("\n=== Summary of All Models ===\n")

for name, (dice, iou, precision, recall) in results.items():
    print(f"{name:20} -> Dice: {dice:.4f}, IoU: {iou:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")